# Tutorial 5: Running a DKG Ceremony


**Audience:** Completed Tutorials 1-4

**Prerequisites:** Scalars, points, secret sharing

**Learning goals:**
- Run a complete distributed key generation ceremony
- Understand every phase: polynomial generation, share distribution, aggregation
- See the Participant-to-module mapping


## What this notebook covers

1. Creating participants
2. Phase 1: Polynomial generation and proof of knowledge
3. Phase 2: Share distribution and verification
4. Phase 3: Share aggregation and public key derivation
5. Group commitments and verification shares
6. Behind the scenes: direct module-level calls
7. Exercise: Lagrange reconstruction of the group secret


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Participant, Scalar, Point, G, Q
from frost import keygen
from frost.polynomial import generate_polynomial, evaluate_polynomial
from frost.lagrange import lagrange_coefficient


## Create Participants

We create a 2-of-3 threshold scheme: 3 participants, any 2 can sign.


In [ ]:
threshold = 2
n = 3
participants = [Participant(index=i, threshold=threshold, participants=n) for i in range(1, n + 1)]
print(f"Created {n} participants with threshold {threshold}")
for p in participants:
    print(f"  Participant {p.index}")


## Phase 1: Polynomial Generation

Each participant generates a random polynomial and computes:
- Coefficient commitments (public points)
- A proof of knowledge of their secret (constant term)


In [ ]:
for p in participants:
    p.init_keygen()

# Each participant now has:
# - coefficients (secret!)
# - coefficient_commitments (public)
# - proof_of_knowledge (public)
p1 = participants[0]
print(f"Participant 1 commitments: {len(p1.coefficient_commitments)} points")
print(f"Participant 1 proof: (R, \u03bc)")


## Proof Verification

Each participant verifies every other participant's proof of knowledge.
This prevents a malicious participant from submitting garbage.


In [ ]:
for verifier in participants:
    for prover in participants:
        if verifier.index == prover.index:
            continue
        valid = verifier.verify_proof_of_knowledge(
            prover.proof_of_knowledge,
            prover.coefficient_commitments[0],
            prover.index,
        )
        print(f"  P{verifier.index} verifies P{prover.index}'s proof: {valid}")


## Phase 2: Share Distribution

Each participant evaluates their polynomial at every other participant's
index to create shares. Participant i sends f_i(j) to participant j.


In [ ]:
for p in participants:
    p.generate_shares()

# The share matrix: p.shares[j-1] is what participant p sends to participant j
for p in participants:
    print(f"P{p.index} shares: [{', '.join(str(s)[:8] + '...' for s in p.shares)}]")


## Share Verification

Each participant verifies received shares against the sender's public
coefficient commitments. This is Feldman's Verifiable Secret Sharing:
share·G should equal the expected combination of commitment points.


In [ ]:
for receiver in participants:
    for sender in participants:
        if receiver.index == sender.index:
            continue
        share = sender.shares[receiver.index - 1]
        valid = receiver.verify_share(share, sender.coefficient_commitments, threshold)
        print(f"  P{receiver.index} verifies share from P{sender.index}: {valid}")


## Phase 3: Aggregation

Each participant sums all received shares (including their own) to get
their aggregate share of the group secret.


In [ ]:
for receiver in participants:
    other_shares = tuple(
        sender.shares[receiver.index - 1]
        for sender in participants
        if sender.index != receiver.index
    )
    receiver.aggregate_shares(other_shares)

for p in participants:
    print(f"P{p.index} aggregate share: {str(p.aggregate_share)[:16]}...")


## Derive Public Key

The group public key is the sum of all participants' secret commitments.
Every participant derives the same public key.


In [ ]:
for p in participants:
    other_commitments = tuple(
        other.coefficient_commitments[0]
        for other in participants
        if other.index != p.index
    )
    p.derive_public_key(other_commitments)

# All participants should agree on the same public key
print(f"P1 public key: ({participants[0].public_key.x}, ...)")
print(f"All agree? {all(p.public_key == participants[0].public_key for p in participants)}")


## Group Commitments and Verification Shares

Group commitments are the element-wise sums of all coefficient commitments.
They allow anyone to derive any participant's public verification share
(Y_i = s_i·G) without knowing the secret share.


In [ ]:
for p in participants:
    other_ccs = tuple(
        other.coefficient_commitments
        for other in participants
        if other.index != p.index
    )
    p.derive_group_commitments(other_ccs)

# Verify: public verification share from aggregate share matches
# derivation from group commitments
for p in participants:
    direct = p.public_verification_share()
    derived = p.derive_public_verification_share(p.group_commitments, p.index, threshold)
    print(f"P{p.index}: direct Y_i == derived Y_i? {direct == derived}")


## Behind the Scenes

The Participant class delegates to protocol modules. Here we redo the
key derivation using direct module calls to show what happens internally.


In [ ]:
# Direct module-level DKG (same result as above)
coeffs_1 = tuple(Scalar(c) for c in participants[0].coefficients)
coeffs_2 = tuple(Scalar(c) for c in participants[1].coefficients)
coeffs_3 = tuple(Scalar(c) for c in participants[2].coefficients)

# Public key = sum of all secret commitments (constant term commitments)
cc1 = keygen.compute_coefficient_commitments(coeffs_1)
cc2 = keygen.compute_coefficient_commitments(coeffs_2)
cc3 = keygen.compute_coefficient_commitments(coeffs_3)
pk = keygen.derive_public_key(cc1[0], (cc2[0], cc3[0]))

print(f"Module-level public key matches Participant? {pk == participants[0].public_key}")


## Exercise

Verify that any 2-of-3 subset of aggregate shares can reconstruct a
secret whose public key matches the group public key.


In [ ]:
from itertools import combinations

for combo in combinations(range(1, 4), threshold):
    indexes = combo
    secret = Scalar(0)
    for idx in indexes:
        lam = lagrange_coefficient(indexes, idx, 0)
        share = Scalar(participants[idx - 1].aggregate_share)
        secret = secret + lam * share
    pk_check = int(secret) * G
    # Compare x-coordinates (public key might need even-y normalization)
    match = pk_check.x == participants[0].public_key.x
    print(f"Shares {combo}: public key matches? {match}")


## Pitfalls and Extensions

**Pitfall:** The group secret y = ∑(a₀ⱼ) is never computed by any
participant. It exists only implicitly in the aggregate shares.

**Extension:** Try the DKG with different thresholds (3-of-5, 4-of-7).
The protocol is the same; only the polynomial degree changes.
